Author: ZheKai

This version deals with three datasets: crc01 (scRNA-seq, HRA003293), crc03 (scRNA-seq, GSE234804), crc06 (snRNA-seq, GSE183916)

In [1]:
### Import modules
import os
from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc
import scipy as sp
import anndata as ad
import sys
import re
### Import personal Utensils
sys.path.insert(0, "..")
import utils as ut

### Set Data Dir

In [2]:
DataRootDir = Path("../../data")

### Data Curation
Now integrate each individual data \

Unification of metadata
- Glob_CellId (used as index): study name + barcode
- Glob_Sample: study name + sample
- Glob_Patient: study name + patient
- Patient: the name of the patient, use the name in the original study
- Study: study name in ./raw-data
- Tissue: crc or pm

It is decided that in Anndata, .X stores log transformed data, while .layers\['counts'] stores raw counts

Besides, we have to unify gene symbols. Since projecting ENSEMBL ID to gene symbol loses 

---
!!! for crc01, patient marked with 'l' means that liver metastasis is also collected

#### Step one: read in and unify data type and meta data(obs)

In [3]:
# Read-in files
CurrentDir = DataRootDir / "crc01"
adata_crc01 = sc.read_h5ad(CurrentDir / "HRA003293.h5ad")

### crc01 data
adata_crc01.X = ut.to_float32(adata_crc01.layers["data"])
del adata_crc01.layers["data"]
adata_crc01.layers["counts"] = ut.to_int32_counts(
    adata_crc01.layers["counts"]
)

# modify obs
crc01_obs = adata_crc01.obs.copy()
obs_list = ['orig.ident','sample','cells','tissue','patient','tissuegroup','tumorgroup','site']
crc01_obs = crc01_obs[obs_list]
crc01_obs['Study'] = 'crc01'
crc01_obs = crc01_obs.rename(columns={"patient": "Patient","tissue":"Tissue","sample":"Sample"})
crc01_obs['Glob_CellID'] = crc01_obs['Study'] + '-' + crc01_obs['cells']
crc01_obs = crc01_obs.drop(columns=["orig.ident",'tissuegroup', 'tumorgroup', 'site',"cells"])
crc01_obs['Glob_Sample'] = crc01_obs['Study'] + '-' + crc01_obs['Sample']
crc01_obs['Glob_Patient'] = crc01_obs['Study'] + '-' + crc01_obs['Patient']
crc01_obs['Tech'] = 'scRNA'
crc01_obs = crc01_obs.set_index("Glob_CellID", drop=False)
adata_crc01.obs = crc01_obs.copy()
del crc01_obs
# rule out liver metastasis
adata_crc01 = adata_crc01[adata_crc01.obs["Tissue"] != "lm"].copy()
# reset analysis
adata_crc01.obsm = {}
adata_crc01.obsp = {}
adata_crc01.var = pd.DataFrame(index=adata_crc01.var.index)

In [ ]:
### crc03 data (scRNA-seq, GSE234804)
CurrentDir = DataRootDir / "crc03/GSE234804"
FileList = sorted(
    file for file in os.listdir(CurrentDir) if file.endswith(".h5ad")
)
adata_list = []

for file in FileList:
    name = Path(file).stem.split("_")[1]  # e.g. "GSM7474991_CRC2_new" -> "CRC2"
    adata = sc.read_h5ad(CurrentDir / file)

    # Clean up layers
    for layer_name in list(adata.layers.keys()):
        if layer_name == "scale.data":
            del adata.layers[layer_name]
            continue
        elif layer_name == "counts":
            adata.layers[layer_name] = ut.to_int32_counts(
                adata.layers[layer_name]
            )
            continue
        elif layer_name == "data":
            adata.layers[layer_name] = sp.sparse.csr_matrix(
                ut.to_float32(adata.layers[layer_name])
            )
            continue

    adata.X = adata.layers["data"]
    del adata.layers["data"]

    # Build metadata
    adata.obs["Study"] = "crc03"
    adata.obs["Sample"] = name
    adata.obs["Tech"] = "scRNA"
    adata.obs["Glob_CellID"] = "crc03-" + name + "_" + adata.obs.index
    adata.obs["Glob_Sample"] = "crc03-" + name

    prefix, num = re.findall(r"(\w+)(\d+)", name)[0]
    if prefix == "PC":
        tissue = "pm"
    elif prefix == "CRC":
        tissue = "crc"
    else:
        raise ValueError(f"Unknown tissue prefix: {prefix}")

    patient = "patient" + num
    adata.obs["Tissue"] = tissue
    adata.obs["Patient"] = patient
    adata.obs["Glob_Patient"] = "crc03-" + adata.obs["Patient"]
    adata.obs = adata.obs.set_index("Glob_CellID", drop=False)

    adata_list.append(adata)

adata_crc03 = ad.concat(
    adata_list,
    join="outer",
    merge="first",
    uns_merge=None,
)

# Reset analysis artifacts, keep var (contains gene_ids)
adata_crc03.obsm = {}
adata_crc03.obsp = {}

In [5]:
# AIGC-generated ---
### crc06 data (snRNA-seq, GSE183916)
adata_crc06 = sc.read_h5ad(DataRootDir / "crc06/GSE183916/GSE183916.h5ad")

adata_crc06.X = sp.sparse.csr_matrix(
    ut.to_float32(adata_crc06.layers["data"])
)
del adata_crc06.layers["data"]
adata_crc06.layers["counts"] = ut.to_int32_counts(
    adata_crc06.layers["counts"]
)

crc06_obs = adata_crc06.obs.copy()
crc06_obs["Study"] = "crc06"
crc06_obs = crc06_obs.rename(columns={
    "pat_id": "Patient",
    "loc": "Tissue",
})
crc06_obs["Tissue"] = crc06_obs["Tissue"].map({
    "metastasis": "pm",
    "primary": "crc",
})
crc06_obs["Sample"] = crc06_obs["Patient"] + "_" + crc06_obs["Tissue"]
crc06_obs["Tech"] = "snRNA"
crc06_obs["Glob_CellID"] = (
    "crc06-" + crc06_obs["Sample"] + "_" + crc06_obs.index
)
crc06_obs["Glob_Sample"] = "crc06-" + crc06_obs["Sample"]
crc06_obs["Glob_Patient"] = "crc06-" + crc06_obs["Patient"]
crc06_obs = crc06_obs.drop(columns=["orig.ident", "nCount_RNA", "nFeature_RNA"])
crc06_obs = crc06_obs.set_index("Glob_CellID", drop=False)
adata_crc06.obs = crc06_obs.copy()
del crc06_obs

adata_crc06.obsm = {}
adata_crc06.obsp = {}
adata_crc06.var = pd.DataFrame(index=adata_crc06.var.index)
# AIGC

In [ ]:
print('Preview Data')
print('-----')
print(ut.preview_adata(adata_crc01))
print(ut.preview_adata(adata_crc03))
print(ut.preview_adata(adata_crc06))
print('-----')
print('type of crc01: X {0} , raw counts {1}'.format(type(adata_crc01.X), type(adata_crc01.layers["counts"])))
print('type of crc03: X {0} , raw counts {1}'.format(type(adata_crc03.X), type(adata_crc03.layers["counts"])))
print('type of crc06: X {0} , raw counts {1}'.format(type(adata_crc06.X), type(adata_crc06.layers["counts"])))
print('-----')
print('data type of crc01: X {0} , raw counts {1}'.format(adata_crc01.X.dtype, adata_crc01.layers["counts"].dtype))
print('data type of crc03: X {0} , raw counts {1}'.format(adata_crc03.X.dtype, adata_crc03.layers["counts"].dtype))
print('data type of crc06: X {0} , raw counts {1}'.format(adata_crc06.X.dtype, adata_crc06.layers["counts"].dtype))

#### Step two: Data Integration

In [ ]:
# Merge crc01, crc03, crc06 by shared gene symbols
adata_crc01.var['gene_name'] = adata_crc01.var_names
adata_crc03.var['gene_name'] = adata_crc03.var_names
adata_crc06.var['gene_name'] = adata_crc06.var_names
adata_crc01.var['gene_id'] = ""
adata_crc03.var = adata_crc03.var.rename(columns={"gene_ids": "old_gene_id"})

adata_merged = ad.concat(
    [adata_crc03, adata_crc06, adata_crc01],
    join="outer",
    merge="first",
    uns_merge=None,
)

ut.preview_adata(adata_merged)
print(f"X dtype: {adata_merged.X.dtype}")
print(f"counts dtype: {adata_merged.layers['counts'].dtype}")

In [8]:
adata_merged.var

,gene_name,gene_id
A1BG,A1BG,
A1BG-AS1,A1BG-AS1,NaN
A1CF,A1CF,
A2M,A2M,
A2M-AS1,A2M-AS1,NaN
...,...,...
ZYG11A,ZYG11A,NaN
ZYG11B,ZYG11B,
ZYX,ZYX,
ZZEF1,ZZEF1,


#### Step three: unify vars
It is decided that gene symbol will be used as index, and ENSG id will be kept as back-ups
Format:
|gene_name|gene_id|
|-----|-----|
|gene A| ENSGXXXX|


Version: GRCh38.111


In [9]:
# Map merged gene symbols to GRCh38.111 ENSG IDs
mapping_path = Path.home() / "t_rna/database/GRCh38.111/gene_id_gene_name.csv"
gene_mapping = pd.read_csv(mapping_path, dtype=str)
gene_mapping["gene_id"] = gene_mapping["gene_id"].str.replace(
    r"\.\d+$", "", regex=True
)

symbol_to_gene_ids = (
    gene_mapping.dropna(subset=["gene_name", "gene_id"])
    .groupby("gene_name")["gene_id"]
    .agg(lambda ids: ",".join(sorted(ids.unique())))
)

adata_merged.var["gene_id"] = adata_merged.var["gene_name"].map(
    symbol_to_gene_ids
)

mapped = adata_merged.var["gene_id"].notna()
multiple = adata_merged.var["gene_id"].str.contains(",", na=False)
adata_merged_unmapped_gene_names = adata_merged.var.index[~mapped].tolist()

print("adata_merged GRCh38.111 mapping")
print(f"mapped: {mapped.sum():,} / {adata_merged.n_vars:,} ({mapped.mean():.2%})")
print(f"cannot map to GRCh38.111: {(~mapped).sum():,}")
print(f"multiple ENSG IDs: {multiple.sum():,}")
print(
    "unmapped gene-name variable: "
    "adata_merged_unmapped_gene_names"
)

if "old_gene_id" in adata_merged.var.columns:
    old_ids = adata_merged.var["old_gene_id"].str.replace(
        r"\.\d+$", "", regex=True
    )
    new_ids = adata_merged.var["gene_id"]
    both_ids_present = old_ids.notna() & new_ids.notna()
    old_id_in_new_mapping = pd.Series(
        [
            pd.notna(old_id) and pd.notna(new_id) and old_id in new_id.split(",")
            for old_id, new_id in zip(old_ids, new_ids)
        ],
        index=adata_merged.var_names,
    )

    adata_merged_old_id_mismatch = adata_merged.var.loc[
        both_ids_present & ~old_id_in_new_mapping,
        ["gene_name", "old_gene_id", "gene_id"],
    ].copy()

    print("\nOld-to-GRCh38.111 comparison")
    print(f"both old and new IDs present: {both_ids_present.sum():,}")
    print(f"old IDs present in new mapping: {old_id_in_new_mapping.sum():,}")
    print(f"old/new ID mismatches: {len(adata_merged_old_id_mismatch):,}")
    print("mismatch table: adata_merged_old_id_mismatch")
else:
    print("\nOld-to-GRCh38.111 comparison: skipped (no old_gene_id column in adata_merged)")

# Add gene biotype as Class (GRCh38.111)
biotype_path = Path.home() / "t_rna/database/GRCh38.111/gene_id_gene_name_gene_biotype.csv"
biotype_mapping = pd.read_csv(biotype_path, dtype=str)
biotype_mapping["gene_id"] = biotype_mapping["gene_id"].str.replace(
    r"\.\d+$", "", regex=True
)

symbol_to_biotype = (
    biotype_mapping.groupby("gene_name")["gene_biotype"]
    .agg(lambda b: ",".join(sorted(b.unique())))
)

adata_merged.var["Class"] = adata_merged.var["gene_name"].map(
    symbol_to_biotype
)

mapped_class = adata_merged.var["Class"].notna()
multiple_class = adata_merged.var["Class"].str.contains(",", na=False)
print("\nClass (gene biotype) mapping")
print(f"Class mapped: {mapped_class.sum():,} / {adata_merged.n_vars:,} ({mapped_class.mean():.2%})")
print(f"multiple biotypes: {multiple_class.sum():,}")
print(adata_merged.var["Class"].value_counts().head(10))

adata_merged GRCh38.111 mapping
mapped: 22,739 / 34,106 (66.67%)
cannot map to GRCh38.111: 11,367
multiple ENSG IDs: 60
unmapped gene-name variable: adata_merged_unmapped_gene_names

Old-to-GRCh38.111 comparison: skipped (no old_gene_id column in adata_merged)

Class (gene biotype) mapping
Class mapped: 22,739 / 34,106 (66.67%)
multiple biotypes: 19
Class
protein_coding     18727
lncRNA              3341
IG_V_pseudogene      184
IG_V_gene            142
TR_V_gene            106
TR_J_gene             79
IG_D_gene             37
TR_V_pseudogene       33
IG_J_gene             18
IG_C_gene             14
Name: count, dtype: int64


In [10]:
adata_merged.var

,gene_name,gene_id,Class
A1BG,A1BG,ENSG00000121410,protein_coding
A1BG-AS1,A1BG-AS1,ENSG00000268895,lncRNA
A1CF,A1CF,ENSG00000148584,protein_coding
A2M,A2M,ENSG00000175899,protein_coding
A2M-AS1,A2M-AS1,ENSG00000245105,lncRNA
...,...,...,...
ZYG11A,ZYG11A,ENSG00000203995,protein_coding
ZYG11B,ZYG11B,ENSG00000162378,protein_coding
ZYX,ZYX,ENSG00000159840,protein_coding
ZZEF1,ZZEF1,ENSG00000074755,protein_coding


### Data Export

In [11]:
export_dir = Path("../../result/curated/merged.h5ad")
adata_merged.write_h5ad(export_dir)